Cell 1: Environment Setup & Architecture Definition

In [1]:
# ========================================================
# CELL 1: ENVIRONMENT SETUP & SIAMESE ARCHITECTURE
# ========================================================
!pip install nibabel scikit-image opencv-python-headless

import os
import glob
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import nibabel as nib
from skimage.metrics import structural_similarity as ssim
from sklearn.metrics import mutual_info_score

# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {device}")

# Multi-Scale Skip-Connection Siamese VGG19
class HighResSiameseVGG19(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.eval()

        self.block1 = vgg[:4]    # relu1_2 -> 64 channels
        self.block2 = vgg[4:9]   # relu2_2 -> 128 channels
        self.block3 = vgg[9:18]  # relu3_4 -> 256 channels
        self.block4 = vgg[18:27] # relu4_4 -> 512 channels

        for param in self.parameters():
            param.requires_grad = False

        self.up4 = nn.Sequential(nn.Conv2d(1024, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True))
        self.up3 = nn.Sequential(nn.Conv2d(256 + 512, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True))
        self.up2 = nn.Sequential(nn.Conv2d(128 + 256, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.up1 = nn.Sequential(nn.Conv2d(64 + 128, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.final = nn.Sequential(nn.Conv2d(32, 1, 1), nn.Sigmoid())

    def forward(self, img_a, img_b):
        a1 = self.block1(img_a); b1 = self.block1(img_b)
        a2 = self.block2(a1);    b2 = self.block2(b1)
        a3 = self.block3(a2);    b3 = self.block3(b2)
        a4 = self.block4(a3);    b4 = self.block4(b3)

        d4 = self.up4(torch.cat([a4, b4], dim=1))
        d4_up = F.interpolate(d4, size=a3.shape[-2:], mode='bilinear', align_corners=False)
        d3 = self.up3(torch.cat([d4_up, a3, b3], dim=1))

        d3_up = F.interpolate(d3, size=a2.shape[-2:], mode='bilinear', align_corners=False)
        d2 = self.up2(torch.cat([d3_up, a2, b2], dim=1))

        d2_up = F.interpolate(d2, size=a1.shape[-2:], mode='bilinear', align_corners=False)
        d1 = self.up1(torch.cat([d2_up, a1, b1], dim=1))

        out = self.final(d1)
        return F.interpolate(out, size=img_a.shape[-2:], mode='bilinear', align_corners=False)

print("Cell 1 Complete: Architecture Loaded.")

Executing on device: cuda
Cell 1 Complete: Architecture Loaded.


Cell 2: Loss Function Definition

In [2]:
# ========================================================
# CELL 2: ADVANCED COMPOUND LOSS FUNCTION
# ========================================================
class AdvancedFusionLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss()

    def gradient_loss(self, pred, target):
        dx_p = torch.abs(pred[:, :, :, :-1] - pred[:, :, :, 1:])
        dx_t = torch.abs(target[:, :, :, :-1] - target[:, :, :, 1:])
        dy_p = torch.abs(pred[:, :, :-1, :] - pred[:, :, 1:, :])
        dy_t = torch.abs(target[:, :, :-1, :] - target[:, :, 1:, :])
        return F.l1_loss(dx_p, dx_t) + F.l1_loss(dy_p, dy_t)

    def forward(self, pred, target):
        return self.bce(pred, target) + 2.0 * self.gradient_loss(pred, target)

print("Cell 2 Complete: Loss Function Defined.")

Cell 2 Complete: Loss Function Defined.


Cell 3: Expanded Dynamic Augmentation Dataset

In [3]:
# ========================================================
# CELL 3: EXPANDED DYNAMIC TRAINING DATASET
# ========================================================
from skimage import data
import torchvision.transforms.functional as TF

os.makedirs("train_samples", exist_ok=True)
sample_imgs = [
    data.astronaut(), data.camera(), data.coffee(), data.chelsea(),
    data.brick(), data.grass(), data.gravel(), data.page(),
    data.immunohistochemistry(), data.retina(), data.coins(), data.moon()
]

for idx, arr in enumerate(sample_imgs):
    if len(arr.shape) == 2:
        arr = cv2.cvtColor(arr, cv2.COLOR_GRAY2RGB)
    elif arr.shape[2] == 4:
        arr = cv2.cvtColor(arr, cv2.COLOR_RGBA2RGB)
    Image.fromarray(arr).save(f"train_samples/img_{idx}.jpg")

class DynamicAugmentedDataset(Dataset):
    def __init__(self, folder, size=(256, 256), samples_per_epoch=200):
        self.paths = glob.glob(os.path.join(folder, "*.jpg"))
        self.size = size
        self.samples_per_epoch = samples_per_epoch
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return self.samples_per_epoch

    def __getitem__(self, idx):
        path = random.choice(self.paths)
        img = cv2.resize(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB), self.size)
        h, w, _ = img.shape

        # Multi-scale random irregular mask synthesis
        gt_mask = np.zeros((h, w), dtype=np.float32)
        mode = random.choice(['horizontal', 'vertical', 'circle', 'polygon'])

        if mode == 'horizontal':
            split = random.randint(h // 4, 3 * h // 4)
            gt_mask[:split, :] = 1.0
        elif mode == 'vertical':
            split = random.randint(w // 4, 3 * w // 4)
            gt_mask[:, :split] = 1.0
        elif mode == 'circle':
            center = (random.randint(w // 4, 3 * w // 4), random.randint(h // 4, 3 * h // 4))
            radius = random.randint(w // 6, w // 3)
            cv2.circle(gt_mask, center, radius, 1.0, -1)
        else:
            num_pts = random.randint(3, 6)
            pts = np.array([[random.randint(0, w), random.randint(0, h)] for _ in range(num_pts)], np.int32)
            cv2.fillPoly(gt_mask, [pts], 1.0)

        # Dynamic variable blur kernels
        k_a = random.choice([11, 17, 23, 29])
        k_b = random.choice([11, 17, 23, 29])
        blurred_a = cv2.GaussianBlur(img, (k_a, k_a), 0)
        blurred_b = cv2.GaussianBlur(img, (k_b, k_b), 0)

        mask_3d = np.repeat(np.expand_dims(gt_mask, 2), 3, 2)
        img_a = (img * mask_3d + blurred_a * (1.0 - mask_3d)).astype(np.uint8)
        img_b = (blurred_b * mask_3d + img * (1.0 - mask_3d)).astype(np.uint8)

        # Convert to tensors
        t_a = self.normalize(TF.to_tensor(Image.fromarray(img_a)))
        t_b = self.normalize(TF.to_tensor(Image.fromarray(img_b)))
        t_gt = torch.tensor(gt_mask, dtype=torch.float32).unsqueeze(0)

        return t_a, t_b, t_gt

train_loader = DataLoader(DynamicAugmentedDataset("train_samples", samples_per_epoch=400), batch_size=8, shuffle=True)
print("Cell 3 Complete: Dynamic 100-Epoch Dataset Loader Configured.")

Cell 3 Complete: Dynamic 100-Epoch Dataset Loader Configured.


Cell 4: 100-Epoch Fine-Tuning with Differential Learning Rates

In [4]:
# ========================================================
# CELL 4: 100-EPOCH FINE-TUNING (UNFROZEN BLOCK4 + DECODER)
# ========================================================
model = HighResSiameseVGG19().to(device)

# Freeze lower blocks, unfreeze block4 for domain adaptation
for p in model.block1.parameters(): p.requires_grad = False
for p in model.block2.parameters(): p.requires_grad = False
for p in model.block3.parameters(): p.requires_grad = False
for p in model.block4.parameters(): p.requires_grad = True

criterion = AdvancedFusionLoss()

# Differential Learning Rates: Low LR for backbone, Standard LR for decoder
params = [
    {'params': model.block4.parameters(), 'lr': 1e-4},
    {'params': model.up4.parameters(), 'lr': 1e-3},
    {'params': model.up3.parameters(), 'lr': 1e-3},
    {'params': model.up2.parameters(), 'lr': 1e-3},
    {'params': model.up1.parameters(), 'lr': 1e-3},
    {'params': model.final.parameters(), 'lr': 1e-3}
]

optimizer = torch.optim.Adam(params, weight_decay=1e-5)
epochs = 100
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

print("Starting 100-Epoch Fine-Tuning...")
model.train()
for epoch in range(epochs):
    loss_acc = 0.0
    for a, b, gt in train_loader:
        a, b, gt = a.to(device), b.to(device), gt.to(device)
        optimizer.zero_grad()

        # Symmetric bidirectional pass
        pred_ab = model(a, b)
        loss_ab = criterion(pred_ab, gt)

        pred_ba = model(b, a)
        loss_ba = criterion(pred_ba, 1.0 - gt)

        total_loss = (loss_ab + loss_ba) / 2.0
        total_loss.backward()
        optimizer.step()
        loss_acc += total_loss.item()

    scheduler.step()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{epochs}] - Loss: {loss_acc/len(train_loader):.4f} - LR: {scheduler.get_last_lr()[0]:.6f}")

torch.save(model.state_dict(), "vgg19_trained_100ep.pth")
print("Cell 4 Complete: 100-Epoch Weights Successfully Saved!")

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:07<00:00, 79.8MB/s]


Starting 100-Epoch Fine-Tuning...
Epoch [001/100] - Loss: 0.3008 - LR: 0.000100
Epoch [010/100] - Loss: 0.0202 - LR: 0.000098


KeyboardInterrupt: 

Cell 5: Benchmark Utilizing the Trained Siamese Weights

In [ ]:
# ========================================================
# CELL 5: BENCHMARK WITH TRAINED 100-EPOCH SIAMESE WEIGHTS
# ========================================================

# Load trained weights into inference model
trained_fusion_model = HighResSiameseVGG19().to(device)
trained_fusion_model.load_state_dict(torch.load("vgg19_trained_100ep.pth", map_location=device))
trained_fusion_model.eval()

input_a = "ct.nii.gz"
input_b = "mr.nii.gz"
is_nifti = any(input_a.endswith(ext) for ext in ['.nii', '.nii.gz'])

def fuse_2d_trained_model(img_a_u8, img_b_u8):
    is_gray = (len(img_a_u8.shape) == 2)
    img_a_rgb = cv2.cvtColor(img_a_u8, cv2.COLOR_GRAY2RGB) if is_gray else img_a_u8
    img_b_rgb = cv2.cvtColor(img_b_u8, cv2.COLOR_GRAY2RGB) if is_gray else img_b_u8

    h, w, _ = img_a_rgb.shape
    np_a = img_a_rgb.astype(np.float32)
    np_b = img_b_rgb.astype(np.float32)

    t_a = vgg_transform(Image.fromarray(img_a_rgb)).unsqueeze(0).to(device)
    t_b = vgg_transform(Image.fromarray(img_b_rgb)).unsqueeze(0).to(device)

    # 1. Forward pass through the 100-epoch trained Siamese network
    with torch.no_grad():
        pred_mask = trained_fusion_model(t_a, t_b).squeeze().cpu().numpy()

    # 2. Strict decision assignment with minimal boundary cleanup
    clean_mask = (pred_mask >= 0.5).astype(np.float32)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    clean_mask = cv2.morphologyEx(clean_mask, cv2.MORPH_CLOSE, kernel)

    weight_3d = np.repeat(clean_mask[:, :, None], 3, axis=2)
    fused_rgb = np.clip(np_a * weight_3d + np_b * (1.0 - weight_3d), 0, 255).astype(np.uint8)

    return cv2.cvtColor(fused_rgb, cv2.COLOR_RGB2GRAY) if is_gray else fused_rgb

# Run inference and calculate metrics
if is_nifti:
    nii_a = nib.load(input_a); nii_b = nib.load(input_b)
    data_a, data_b = nii_a.get_fdata(), nii_b.get_fdata()
    norm_a = ((data_a - data_a.min()) / (data_a.max() - data_a.min() + 1e-8) * 255.0).astype(np.uint8)
    norm_b = ((data_b - data_b.min()) / (data_b.max() - data_b.min() + 1e-8) * 255.0).astype(np.uint8)
    depth = norm_a.shape[-1] if len(norm_a.shape) == 3 else 1

    fused_vol_orig = np.zeros_like(norm_a)
    fused_vol_trained = np.zeros_like(norm_a)

    for z in range(depth):
        sl_a = norm_a[:, :, z] if len(norm_a.shape) == 3 else norm_a
        sl_b = norm_b[:, :, z] if len(norm_b.shape) == 3 else norm_b
        fused_vol_orig[:, :, z] = fuse_2d_baseline(sl_a, sl_b)
        fused_vol_trained[:, :, z] = fuse_2d_trained_model(sl_a, sl_b)

    mid = depth // 2
    vis_a, vis_b = norm_a[:, :, mid], norm_b[:, :, mid]
    vis_orig, vis_trained = fused_vol_orig[:, :, mid], fused_vol_trained[:, :, mid]
else:
    img_a = Image.open(input_a).convert("RGB")
    img_b = Image.open(input_b).convert("RGB")
    if img_a.size != img_b.size:
        img_b = img_b.resize(img_a.size)
    vis_a = np.array(img_a, dtype=np.uint8)
    vis_b = np.array(img_b, dtype=np.uint8)
    vis_orig = fuse_2d_baseline(vis_a, vis_b)
    vis_trained = fuse_2d_trained_model(vis_a, vis_b)

m_orig = calculate_metrics(vis_a, vis_b, vis_orig)
m_trained = calculate_metrics(vis_a, vis_b, vis_trained)

comp_df = pd.DataFrame({
    "Metric": [
        "Entropy (EN)", "Standard Deviation (SD)", "Spatial Frequency (SF)",
        "Average Gradient (AG)", "Mutual Information (MI)", "Structural Similarity (SSIM)", "Correlation (CC)"
    ],
    "Original Pretrained VGG19": np.round(m_orig, 4),
    "100-Epoch Trained VGG19": np.round(m_trained, 4),
    "Improvement (%)": [f"{((f - o) / o * 100):+.2f}%" for o, f in zip(m_orig, m_trained)],
    "Optimal Trend": ["Higher (↑)"] * 7
})

cmap_val = 'gray' if len(vis_a.shape) == 2 else None
plt.figure(figsize=(20, 5), dpi=200)
plt.subplot(1, 4, 1); plt.imshow(vis_a, cmap=cmap_val); plt.title("Source A / Mid-Slice"); plt.axis("off")
plt.subplot(1, 4, 2); plt.imshow(vis_b, cmap=cmap_val); plt.title("Source B / Mid-Slice"); plt.axis("off")
plt.subplot(1, 4, 3); plt.imshow(vis_orig, cmap=cmap_val); plt.title("Original VGG19 Baseline"); plt.axis("off")
plt.subplot(1, 4, 4); plt.imshow(vis_trained, cmap=cmap_val); plt.title("100-Epoch Trained Siamese Output", fontweight='bold'); plt.axis("off")
plt.tight_layout()
plt.show()

display(comp_df)